# Splitting Zone

This notebook is responsible of downloading the manifest file generated in the labelling zone and:
- Generating the sample according to different criterias if needed
- Generating two new manifests corresponding to the train and test split.
 

## Preparing new buckets

In [10]:
import boto3
from botocore.exceptions import ClientError
import os
from dotenv import load_dotenv

load_dotenv()
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
minio_url = "http://" + os.getenv("S3_API_ENDPOINT")


minio_client = boto3.client(
    "s3",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=minio_url
)

manifest_name = "dataset_labelled.json"

In [11]:
new_bucket = "splitting-zone"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code in ['BucketAlreadyExists', 'BucketAlreadyOwnedByYou']:
        print(f"Bucket '{new_bucket}' already exists")
    else:
        print(f"Error creating bucket: {e}")

Bucket 'splitting-zone' already exists


## Analyze complete dataset

In [12]:
import pandas as pd

source_bucket = "labelling-zone"
minio_client.download_file(source_bucket, manifest_name, manifest_name)

df = pd.read_json(manifest_name)
print(df.columns)
print(f"Dataset shape: {df.shape}")
print(df.describe())
print(df["text"].value_counts())

Index(['image', 'text', 'l2_distance', 'cosine_similarity'], dtype='object')
Dataset shape: (100, 4)
       l2_distance  cosine_similarity
count   100.000000         100.000000
mean      1.394165           0.302918
std       0.034663           0.017332
min       1.317803           0.246140
25%       1.374318           0.293748
50%       1.394006           0.302997
75%       1.412504           0.312841
max       1.507720           0.341099
text
texts/melanoma_0_0_74.txt                  27
texts/melanoma_0_0_68.txt                  14
texts/melanoma_0_0_24.txt                   9
texts/basal_cell_carcinoma_0_0_47.txt       8
texts/skin_cancer_0_0_50.txt                7
texts/basal_cell_carcinoma_0_0_13.txt       5
texts/melanoma_0_0_73.txt                   3
texts/basal_cell_carcinoma_0_0_3.txt        3
texts/melanoma_0_0_155.txt                  2
texts/melanoma_0_0_70.txt                   2
texts/basal_cell_carcinoma_0_0_9.txt        2
texts/melanoma_0_0_164.txt                  2


## Generate Split

To perform the split, we will generate a split of 80/20% for train and test, trying to preserve the same proportions in both cases. This can be done easily using the train_test_split function from sklearn.

In [16]:
from sklearn.model_selection import train_test_split

train_manifest_name = "dataset_train.json"
test_manifest_name = "dataset_test.json"

train_df, test_df = train_test_split(df, test_size=0.2, random_state=0)
train_df.to_json(train_manifest_name)
test_df.to_json(test_manifest_name)

minio_client.upload_file(train_manifest_name, new_bucket, train_manifest_name)
minio_client.upload_file(test_manifest_name, new_bucket, test_manifest_name)

In [17]:
print(train_df.head())
print(train_df["text"].value_counts())

                      image                                   text  \
43  images/ISIC_0029220.png              texts/melanoma_0_0_74.txt   
62  images/ISIC_0030492.png  texts/basal_cell_carcinoma_0_0_47.txt   
3   images/ISIC_0025118.png              texts/melanoma_0_0_74.txt   
71  images/ISIC_0031981.png              texts/melanoma_0_0_74.txt   
45  images/ISIC_0029475.png              texts/melanoma_0_0_74.txt   

    l2_distance  cosine_similarity  
43     1.409959           0.295021  
62     1.385651           0.307174  
3      1.369394           0.315303  
71     1.421296           0.289352  
45     1.380617           0.309692  
text
texts/melanoma_0_0_74.txt                  19
texts/melanoma_0_0_68.txt                  10
texts/melanoma_0_0_24.txt                   9
texts/basal_cell_carcinoma_0_0_47.txt       7
texts/skin_cancer_0_0_50.txt                5
texts/basal_cell_carcinoma_0_0_13.txt       5
texts/basal_cell_carcinoma_0_0_3.txt        3
texts/melanoma_0_0_73.txt     

In [18]:
print(test_df.head())
print(test_df["text"].value_counts())

                      image                       text  l2_distance  \
26  images/ISIC_0027209.png  texts/melanoma_0_0_68.txt     1.370836   
86  images/ISIC_0033103.png  texts/melanoma_0_0_68.txt     1.329833   
2   images/ISIC_0024853.png  texts/melanoma_0_0_74.txt     1.401759   
55  images/ISIC_0030055.png  texts/melanoma_0_0_74.txt     1.420965   
75  images/ISIC_0032150.png  texts/melanoma_0_0_74.txt     1.365214   

    cosine_similarity  
26           0.314582  
86           0.335083  
2            0.299120  
55           0.289517  
75           0.317393  
text
texts/melanoma_0_0_74.txt                  8
texts/melanoma_0_0_68.txt                  4
texts/skin_cancer_0_0_50.txt               2
texts/skin_cancer_0_0_32.txt               1
texts/squamous_cell_carcinoma_0_0_4.txt    1
texts/melanoma_0_0_57.txt                  1
texts/melanoma_0_0_164.txt                 1
texts/actinic_keratosis_0_0_5.txt          1
texts/basal_cell_carcinoma_0_0_47.txt      1
Name: count, dtype: